# Huấn luyện long-range v2.1 trên Google Colab

Notebook này chạy **toàn bộ** phần huấn luyện nặng của mô hình dự báo 1..168 giờ (v2.1) trên Colab thay cho laptop.
Chạy lần lượt từ trên xuống (Runtime → Run all). Mỗi giai đoạn lưu kết quả vào Google Drive, nên nếu Colab bị ngắt
kết nối thì chỉ cần chạy lại: các bước đã xong sẽ được bỏ qua.

Thứ tự: kiểm tra máy → gắn Drive → cài thư viện (runtime tự khởi động lại) → chép dữ liệu → smoke test →
`dev` (chọn mode trên VAL_FIT) → `final` (huấn luyện + hiệu chỉnh trên VAL_CAL) → `spatial` → `test` (chỉ **một lần**) →
nén artifact để tải về.

Lưu ý: nhãn là dữ liệu mô hình Open-Meteo, không phải quan trắc trạm; khoảng 80 % không có nghĩa là độ chính xác 80 %.

In [ ]:
# 1. Kiểm tra runtime (Python, CPU, RAM)
import os, platform, sys
assert sys.version_info >= (3, 12), ("Cần Python >= 3.12 (numpy/scipy đã ghim yêu cầu 3.12): "
                                      "Runtime → Change runtime type → chọn phiên bản runtime mới nhất")
print("Python", platform.python_version(), "| CPU:", os.cpu_count())
mem_kb = int([l for l in open("/proc/meminfo") if l.startswith("MemTotal")][0].split()[1])
ram_gb = mem_kb / 1024 ** 2
print(f"RAM: {ram_gb:.1f} GB")
if ram_gb < 12:
    print("CẢNH BÁO: RAM < 12 GB. Nên chọn Runtime → Change runtime type → High-RAM (nếu có).")

In [ ]:
# 2. Gắn Google Drive; thư mục đã tải lên phải là MyDrive/sky_weather_colab
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/sky_weather_colab")
for need in ("data/manifest.json", "code.zip", "code.zip.sha256", "requirements-colab.txt"):
    assert (ROOT / need).exists(), f"Thiếu {ROOT / need}: hãy tải thư mục colab_upload lên Drive và đổi tên thành sky_weather_colab"
print("Drive OK:", sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# 3. Cài đúng phiên bản thư viện rồi tự khởi động lại runtime (chỉ cần ở lần đầu của mỗi phiên)
import importlib.metadata as md, os
REQ = "/content/drive/MyDrive/sky_weather_colab/requirements-colab.txt"
pins = dict(l.strip().split("==") for l in open(REQ) if "==" in l and not l.startswith("#"))
have = {}
for name in pins:
    try:
        have[name] = md.version(name)
    except md.PackageNotFoundError:
        have[name] = None
if all(have[n] == v for n, v in pins.items()):
    print("Đã đúng phiên bản, không cần cài lại:", pins)
else:
    rc = os.system(f"pip install -q -r {REQ}")
    if rc != 0:
        raise SystemExit("pip không cài được đúng phiên bản (có thể Colab chưa có wheel cho Python này). "
                         "KHÔNG tự đổi phiên bản: sửa requirements-colab.txt có chủ đích rồi chạy lại.")
    print("Đã cài xong. Runtime sẽ khởi động lại; sau đó chọn ô 4 và Runtime → Run after.")
    os.kill(os.getpid(), 9)

In [ ]:
# 4. Kiểm tra phiên bản sau khi khởi động lại (không chấp nhận lệch phiên bản âm thầm)
import importlib.metadata as md
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/sky_weather_colab")
pins = dict(l.strip().split("==") for l in open(ROOT / "requirements-colab.txt") if "==" in l and not l.startswith("#"))
bad = {n: (v, md.version(n)) for n, v in pins.items() if md.version(n) != v}
assert not bad, f"Lệch phiên bản (cần, đang có): {bad}"
print("Phiên bản khớp:", pins)

In [ ]:
# 5. Chép dữ liệu + code sang ổ SSD cục bộ của Colab và kiểm tra sha256
import hashlib, json, os, shutil, subprocess, zipfile
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/sky_weather_colab")
WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

if not (WORK / "data" / "manifest.json").exists():
    shutil.copytree(ROOT / "data", WORK / "data")
manifest = json.loads((WORK / "data" / "manifest.json").read_text())
for name, info in manifest["files"].items():
    assert sha(WORK / "data" / name) == info["sha256"], f"sha256 sai: {name}"
expected = (ROOT / "code.zip.sha256").read_text().split()[0]
assert sha(ROOT / "code.zip") == expected, "code.zip bị hỏng khi tải lên"
if not (WORK / "backend").exists():
    with zipfile.ZipFile(ROOT / "code.zip") as z:
        z.extractall(WORK)
os.chdir(WORK)
THREADS = os.cpu_count()
RUNS = ROOT / "runs"
RUNS.mkdir(exist_ok=True)
LEDGER = ROOT / "ledger" / "test_ledger.jsonl"
TRAIN = f"python backend/scripts/train_long_range_v21.py --data {WORK / 'data'} --colab --threads {THREADS}"

def run(cmd):
    print(cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

print(len(manifest["files"]), "tệp dữ liệu OK; code OK; threads =", THREADS)

In [ ]:
# 6. Smoke test toàn chuỗi (dev → final → spatial → test → nén), 3 ô lưới, thư mục tạm trên /content.
#    Dùng ledger RIÊNG của smoke (SMOKE/ledger.jsonl), KHÔNG BAO GIỜ đụng ledger thật trên Drive.
SMOKE = WORK / "smoke"
if SMOKE.exists():
    shutil.rmtree(SMOKE)
S_LEDGER = SMOKE / "ledger.jsonl"
S_ART = SMOKE / "art"
run(f"{TRAIN} --phase dev --smoke --cells 3 --out {SMOKE / 'dev'}")
run(f"{TRAIN} --phase final --smoke --cells 3 --selection {SMOKE / 'dev' / 'dev_selection.json'} "
    f"--ledger {S_LEDGER} --out {S_ART}")
run(f"{TRAIN} --phase spatial --smoke --out {S_ART} --spatial-dir {SMOKE / 'spatial'}")
run(f"{TRAIN} --phase test --smoke --out {S_ART} --spatial-dir {SMOKE / 'spatial'} --ledger {S_LEDGER}")
shutil.make_archive(str(SMOKE / "art_zip"), "zip", root_dir=S_ART)
shutil.rmtree(SMOKE)
print("Smoke OK (dev, final, spatial, test, zip)")

In [ ]:
# 7. Phase dev: so sánh mode × loại booster × năm bắt đầu train trên VAL_FIT -> dev_selection.json (Drive)
DEV = RUNS / "dev"
if (DEV / "dev_selection.json").exists():
    print("Đã có", DEV / "dev_selection.json", "-> bỏ qua (resume)")
else:
    local = WORK / "dev"
    if local.exists():
        shutil.rmtree(local)
    run(f"{TRAIN} --phase dev --out {local}")
    shutil.copytree(local, DEV, dirs_exist_ok=True)
sel = json.loads((DEV / "dev_selection.json").read_text())
print(sel["argmin"], sel["train_start"]["decision"])

In [ ]:
# 8. Phase final: boosters trên train (early stopping VAL_FIT); taper, V3, hiệu chỉnh mưa và conformal trên VAL_CAL.
#    Ghi test_spec.json + parity_cases.json. KHÔNG chấm test ở bước này.
ART = RUNS / "long_range_v21"

def manifest_ok(folder):
    lines = (folder / "MANIFEST.sha256").read_text().splitlines()
    return all((folder / rel).is_file() and sha(folder / rel) == digest
               for digest, rel in (l.split("  ", 1) for l in lines if l.strip()))

if (ART / "MANIFEST.sha256").exists():
    assert manifest_ok(ART), (f"{ART} không khớp MANIFEST.sha256 (chép dở?): xem README mục 'Sự cố', "
                              "đổi tên thư mục đó rồi chạy lại ô này")
    print("Đã có artifact trên Drive (MANIFEST khớp) -> bỏ qua (resume). Không chạy lại final với spec khác.")
else:
    assert not ART.exists(), f"{ART} còn dở dang từ phiên trước: đổi tên thư mục đó rồi chạy lại ô này"
    local = WORK / "long_range_v21"
    if local.exists():
        shutil.rmtree(local)
    run(f"{TRAIN} --phase final --selection {DEV / 'dev_selection.json'} --ledger {LEDGER} --out {local}")
    partial = ART.with_name(ART.name + ".partial")
    if partial.exists():
        shutil.rmtree(partial)
    shutil.copytree(local, partial)   # chép vào .partial rồi mới đổi tên: mất kết nối giữa chừng không để lại
    partial.rename(ART)               # thư mục ART "có MANIFEST nhưng thiếu file"
meta = json.loads((ART / "metadata.json").read_text())
print(meta["model_version"], meta["parity"], meta["resources"])

In [ ]:
# 9. Phase spatial: boosters bỏ-khối-ô (Côn Đảo riêng + 5 khối KMeans), lưu trên Drive, tự resume theo fold
SPATIAL = RUNS / "spatial"
run(f"{TRAIN} --phase spatial --out {ART} --spatial-dir {SPATIAL}")
print(sorted(p.name for p in SPATIAL.iterdir()))

In [ ]:
# 10. Phase test: chấm TEST đúng MỘT lần (ledger toàn cục trên Drive + kiểm tra sha256 của test_spec.json).
#     Trainer ghi mục 'started' vào ledger TRƯỚC khi đọc dữ liệu test, và 'completed' khi xong.
def v21_entries():
    if not LEDGER.exists():
        return []
    return [e for e in map(json.loads, filter(str.strip, LEDGER.read_text().splitlines()))
            if e.get("program") == "long_range_v21"]

status = [e.get("status") for e in v21_entries()]
if "completed" in status:
    print("Test đã được chấm (ledger: completed) -> không chấm lại.")
elif "started" in status:
    raise SystemExit("Ledger ghi test đã BẮT ĐẦU nhưng chưa xong (phiên bị ngắt). Theo quy tắc chấm một lần, "
                     "không được chấm lại: xem README mục 'Sự cố'.")
else:
    run(f"{TRAIN} --phase test --out {ART} --spatial-dir {SPATIAL} --ledger {LEDGER}")
report = json.loads((ART / "test_report.json").read_text())
print(json.dumps(report["criteria"], indent=1)[:4000])

In [ ]:
# 11. Xem hình (skill theo giờ, độ tin cậy mưa theo dải, coverage, đường nhảy)
from IPython.display import Image, display
for p in sorted((ART / "figures").glob("*.png")):
    print(p.name)
    display(Image(filename=str(p)))

In [ ]:
# 12. Nén artifact + sha256, lưu lên Drive và tải về máy
version = json.loads((ART / "metadata.json").read_text())["model_version"]
zip_base = WORK / f"long_range_v21_{version}"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=ART))
digest = sha(zip_path)
sha_file = zip_path.parent / (zip_path.name + ".sha256")
sha_file.write_text(f"{digest}  {zip_path.name}\n")
shutil.copy2(zip_path, ROOT / zip_path.name)
shutil.copy2(sha_file, ROOT / sha_file.name)
print(zip_path.name, digest)
from google.colab import files
files.download(str(zip_path))
files.download(str(sha_file))